# EPANET Pressure Extraction
Runs peak-demand simulations for every province × year and collects pressures at outlet nodes (`-outlet` suffix) and GM nodes (`GM` prefix).

In [ ]:
%pip install -e ../../../

## 1 · System setup

In [ ]:
import os
import numpy as np
from numpy.random import default_rng
RNG = default_rng(12811)
import pandas as pd

from water_futures_battle.core import Settings
from water_futures_battle.jurisdictions import build_state
from water_futures_battle.jurisdictions.entities import Province
from water_futures_battle.sources import build_sources
from water_futures_battle.pumping_stations import build_pumping_infrastructure

settings_desc = {
    'start_year': 2000,
    'end_year': 2024,
    'national_budget': 0,
    'lifeline_volume': 80,
    'seed': 128
}
settings = Settings.from_config(settings_desc)

nl_desc = {
    'name': 'Netherlands',
    'id': 'NL0000',
    'jurisdictions-static_properties': 'jurisdictions-static_properties.xlsx',
    'municipalities-dynamic_properties': 'municipalities-dynamic_properties.xlsx'
}
netherlands = build_state(
    nl_desc,
    data_path='../../jurisdictions/',
    settings=settings
)

pvs_max_daily_dem_df = pd.read_csv(
    '../../jurisdictions/provinces-max_daily_demands-CMD.csv',
    index_col='year'
)
juris_max_hourly_dem_df = pd.read_csv(
    '../../jurisdictions/jurisdictions-max_hourly_demands-CMH.csv',
    index_col='year',
    parse_dates=True
)
juris_max_hourly_dem_df.index = juris_max_hourly_dem_df.index.year

sources_desc = {
    'sources-static_properties': 'sources-static_properties.xlsx',
    'groundwater-dynamic_properties': 'groundwater-dynamic_properties.xlsx',
    'surface_water-dynamic_properties': 'surface_water-dynamic_properties.xlsx',
    'desalination-dynamic_properties': 'desalination-dynamic_properties.xlsx',
    'groundwater-construction_size-max_permit_surplus': 0.3,
    'surface_water-construction_size-min': 0,
    'surface_water-construction_size-max': 0,
    'desalination-construction_size-min': 0,
    'desalination-construction_size-max': 0,
}
all_sources, sources_settings = build_sources(
    sources_desc,
    a_state=netherlands,
    data_path='../../sources/',
    settings=settings
)
sources_lookup = pd.read_excel(
    '../../sources/sources-static_properties.xlsx',
    sheet_name=list(all_sources.types_names),
    index_col='source_id'
)

pumping_desc = {
    'pumping_stations-static_properties': 'pumping_stations/pumping_stations-static_properties.xlsx',
    'pump_options-static_properties': 'pumps/pump_options-static_properties.xlsx',
    'pump_options-dynamic_properties': 'pumps/pump_options-dynamic_properties.xlsx'
}
pump_options, pumping_stations = build_pumping_infrastructure(
    desc=pumping_desc,
    sources=all_sources,
    data_path='../../',
    settings=settings
)

## 2 · Constants

In [ ]:
from epanet_plus import EPyT
from epanet_plus import EpanetConstants

SITUATION_NAME = 'checkpoint_8'
DEST_FOLDER    = f'../../epanet-networks/.debug/'
YEARS          = list(range(2000, 2025))

PROVINCES: list[Province] = list(netherlands.provinces)
pumping_stations_map_sources = {
    ps.bwf_id: ps.source
    for ps in pumping_stations
}
sources_municipality_can_be_open_2025 = set([ 
    s.closest_municipality.effective_entity(when=2025).cbs_id
    for s in all_sources
    if pd.isna(s.activation_date)
])


In [ ]:
print(sources_municipality_can_be_open_2025)

## 3 · Helper functions

In [ ]:
def inp_path(province: Province, year: int) -> str:
    """Return the .inp filepath for a given province and year, raising if absent."""
    folder = f"{DEST_FOLDER}/{province.outfiles_name}/{SITUATION_NAME}"
    filename = f"{province.outfiles_name}-{year}-{SITUATION_NAME}.inp"
    path = os.path.join(folder, filename)
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Network file not found: {path}\n"
            f"  province={province.outfiles_name!r}, year={year}"
        )
    return path


def get_node_pressures(net: EPyT) -> dict[str, float]:
    """
    Run a single-step hydraulic simulation and return a {node_id: pressure}
    mapping for every junction in the network.
    Assumptions: PDA and peak-demand settings are already applied to `net`.
    """
    try:
        net.openH()
        net.initH(EpanetConstants.EN_INITFLOW)
        net.runH()
        pressures = np.round(
            np.array(net.getnodevalues(EpanetConstants.EN_PRESSURE)), 4
        )
        node_ids  = net.get_all_nodes_id()          # list of all node IDs
        net.closeH()
        return dict(zip(node_ids, pressures))
    except Exception:
        net.closeH()
        raise


def filter_outlet_nodes(pressures: dict[str, float]) -> dict[str, float]:
    """Keep only nodes whose ID ends with '-outlet'."""
    return {k: v for k, v in pressures.items() if k.endswith('-outlet')}


def filter_gm_nodes(pressures: dict[str, float]) -> dict[str, float]:
    """Keep only nodes whose ID starts with 'GM'."""
    return {k: v for k, v in pressures.items() if k.startswith('GM')}

## 4 · Main loop

In [ ]:
outlet_rows: list[dict] = []
gm_rows:     list[dict] = []

for province in PROVINCES:
    for year in YEARS:
        path = inp_path(province, year)

        net = EPyT(
            inp_file_in=path,
            use_project=True,
            ignore_error_codes=[4,5,6]    
        )

        try:
            all_pressures = get_node_pressures(net)
        finally:
            net.close()

        # --- outlet nodes ---
        for node_id, pressure in filter_outlet_nodes(all_pressures).items():
            outlet_rows.append({
                'province': province.cbs_id,
                'year':     year,
                'node':     node_id,
                'pressure': pressure,
            })

        # --- GM nodes ---
        for node_id, pressure in filter_gm_nodes(all_pressures).items():
            gm_rows.append({
                'province': province.cbs_id,
                'year':     year,
                'node':     node_id,
                'pressure': pressure,
            })

        print(f"  ✓  {province.outfiles_name}  {year}")

print("\nDone.")

## 5 · Assemble DataFrames

In [ ]:
outlet_df = (
    pd.DataFrame(outlet_rows)
      .set_index(['year', 'node'])
      .sort_index()
)

gm_df = (
    pd.DataFrame(gm_rows)
      .set_index(['year', 'node'])
      .sort_index()
)

print(f"outlet_df : {outlet_df.shape}  — {outlet_df.index.get_level_values('node').nunique()} unique outlet nodes")
print(f"gm_df     : {gm_df.shape}  — {gm_df.index.get_level_values('node').nunique()} unique GM nodes")

display(outlet_df.head(10))

target_rows = []

for (year, node), row in outlet_df.iterrows():
    ps_id = node[:6]
    source = pumping_stations_map_sources.get(ps_id)
    if source is None:
        raise KeyError(f"No pumping station found for ps_id={ps_id!r} (from node={node!r})")
    
    muni_id = source.closest_municipality.effective_cbs_id(when=year)
    target_rows.append({
        'year':    year,
        'muni_id': muni_id,
        'pressure': row['pressure'],
    })

target_heads = (
    pd.DataFrame(target_rows)
      .pivot_table(index='year', columns='muni_id', values='pressure', aggfunc='mean')
      .sort_index()
      .round(4)
)

# --- fill missing 2024 municipalities from gm_df ---
missing = sources_municipality_can_be_open_2025.difference(target_heads.columns)
gm_2024 = (
    gm_df.loc[2024, 'pressure']   # series indexed by node (= muni_id)
            .dropna()
)
for m in missing:
    target_heads[m] = np.nan
    target_heads.loc[2024, m] = (gm_2024[m] + 3).round(4)

target_heads = target_heads.sort_index(axis=1)  # keep columns ordered
target_heads.to_csv('target_heads.csv')
display(target_heads.head())

In [ ]:
display(gm_df.head(10))

### Optional: pivot to wide format
Rows = `(province, year)`, columns = node IDs. Handy for quick inspection.

In [ ]:
outlet_wide = outlet_df['pressure'].unstack('node')
gm_wide     = gm_df['pressure'].unstack('node')

gm_wide.round()